In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("data/creditcard.csv")

In [3]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO TREINO/TESTE
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
from imblearn.combine import SMOTEENN
from collections import Counter

sme = SMOTEENN(random_state=42)
X_train, y_train = sme.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_train))

Resampled dataset shape Counter({1: 218370, 0: 209282})


FLAML (Funcionando no Colab)

In [10]:
# =========================
# INSTALAR FLAML
# =========================
%pip install "flaml[automl]"

# =========================
# IMPORTS
# =========================
from flaml import AutoML

print("Dataset carregado:")
print(df.head())

# =========================
# TREINAMENTO COM FLAML
# =========================
automl = AutoML()

settings = {
    "time_budget": 60,
    "metric": 'accuracy',
    "task": 'classification',
    "log_file_name": "flaml.log",
}

automl.fit(X_train=X_train, y_train=y_train, **settings)

# =========================
# RESULTADOS
# =========================
print("\n===== RESULTADOS =====")
print("Melhor modelo:", automl.best_estimator)
print("Melhores parâmetros:", automl.best_config)
print("Melhor loss:", automl.best_loss)

# =========================
# AVALIAÇÃO
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))

  Using cached flaml-2.7.0-py3-none-any.whl.metadata (14 kB)
  Using cached xgboost-2.1.4-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached flaml-2.7.0-py3-none-any.whl (352 kB)
Using cached xgboost-2.1.4-py3-none-macosx_12_0_arm64.whl (1.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]
Note: you may need to restart the kernel to use updated packages.
Dataset carregado:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.3637

Auto-PyTorch (Não está funcionando no Colab, Auto-PyTorch: python 3.10, Colab: 3.12)

In [ ]:
import sys
from autoPyTorch.api.tabular_classification import TabularClassificationTask

print(sys.executable)
print("Auto-PyTorch works")

In [ ]:
# =========================
# INSTALAR AUTO-PYTORCH
# =========================
#%pip install autoPyTorch==0.2.1 scikit-learn==0.24.2

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from autoPyTorch.api.tabular_classification import TabularClassificationTask

# =========================
# AUTO-PYTORCH
# =========================
automl = TabularClassificationTask()

automl.search(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    optimize_metric='accuracy',
    total_walltime_limit=300,   # tempo total (5 min)
    func_eval_time_limit_secs=50
)

# =========================
# RESULTADOS
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))

TPOT (Não está funcionando no Colab, TPOT: python 3.10, Colab: 3.12)

In [6]:
# =========================
# INSTALAR TPOT NOVO
# =========================
!pip install tpot

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from tpot import TPOTClassifier

# =========================
# TPOT (NOVO)
# =========================
tpot = TPOTClassifier(
    generations=3,
    population_size=10,
    random_state=42,
    n_jobs=1,
    #config_dict='TPOT light'
)

tpot.fit(X_train, y_train)

# =========================
# RESULTADOS
# =========================
y_pred = tpot.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))

/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
Generation:   0%|          | 0/3 [00:00<?, ?it/s]/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Generation:  67%|██████▋   | 2/3 [1:08:23<34:11, 2051.61s/it]
/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Us


===== MÉTRICAS =====
Accuracy: 0.9973
Precision: 0.3739
Recall: 0.8776
F1 Score: 0.9978


/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lannaces/Documents/TCC/.venv310/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


Auto-SkLearn (Não está funcionando no Colab, Auto-SkLearn: python 3.10, Colab: 3.12)

In [ ]:
# =========================
# INSTALAR AUTO-SKLEARN
# =========================
#%pip install autosklearn

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

import autosklearn.classification


# =========================
# AUTO-SKLEARN
# =========================
automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=300,   # tempo total (5 min)
    per_run_time_limit=60,         # tempo por modelo
    memory_limit=4096,             # memória (MB)
    seed=42
)

automl.fit(X_train, y_train)

# =========================
# RESULTADOS
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'), 4))

# =========================
# MODELOS USADOS
# =========================
print("\n===== MODELOS =====")
print(automl.show_models())

LazyPredit (Funcionando no Colab)

In [ ]:
# =========================
# INSTALAR LAZYPREDICT
# =========================
#%pip install lazypredict

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lazypredict.Supervised import LazyClassifier

# =========================
# LAZYPREDICT
# =========================
clf = LazyClassifier(verbose=0, ignore_warnings=True)

models, predictions = clf.fit(X_train, X_test, y_train, y_test)

# =========================
# RESULTADOS
# =========================
print("\n===== RESULTADOS =====")
print(models)

print("\n===== MÉTRICAS =====")
all_metrics = []

for model_name, model in clf.models.items():
    y_pred = model.predict(X_test)

    all_metrics.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, pos_label=1, zero_division=0, average='weighted'),
    })

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values("F1 Score", ascending=False)
    .reset_index(drop=True)
)

print(metrics_df)

ModuleNotFoundError: No module named 'lazypredict.Supervised'

H20 AutoML (Funcionando no Colab)

In [9]:
# =========================
# INSTALAR H2O
# =========================
%pip install h2o

# =========================
# IMPORTS
# =========================
import pandas as pd
import h2o
from h2o.automl import H2OAutoML

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# INICIAR H2O
# =========================
h2o.init()

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PARA H2O
# =========================
train_h2o = h2o.H2OFrame(train_df)
test_h2o = h2o.H2OFrame(test_df)

train_h2o['Class'] = train_h2o['Class'].asfactor()
test_h2o['Class'] = test_h2o['Class'].asfactor()

x = train_h2o.columns[:-1]
y = 'Class'

# =========================
# AUTO ML
# =========================
aml = H2OAutoML(
    max_models=20,
    seed=42,
    balance_classes=True
)

aml.train(x=x, y=y, training_frame=train_h2o)

# =========================
# LEADERBOARD
# =========================
print("\n===== LEADERBOARD =====")
print(aml.leaderboard)

# =========================
# MÉTRICAS DE TODOS OS MODELOS
# =========================
leaderboard = aml.leaderboard.as_data_frame()

resultados = []

for model_id in leaderboard['model_id']:

    model = h2o.get_model(model_id)

    # previsão
    preds = model.predict(test_h2o).as_data_frame()

    y_true = test_df['Class'].values
    y_pred = preds['predict'].astype(int).values

    # métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)

    resultados.append({
        "Modelo": model_id,
        "Tipo": model.algo,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-score": round(f1, 4)
    })

# =========================
# RESULTADO FINAL
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="F1-score", ascending=False))

  Using cached h2o-3.46.0.12.tar.gz (266.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
  Created wheel for h2o: filename=h2o-3.46.0.12-py2.py3-none-any.whl size=266411236 sha256=067ef99d81c45848a7fb823d9446c4c68ce37b2cb1255a95b308b62af6d5ed59
  Stored in directory: /Users/lannaces/Library/Caches/pip/wheels/e5/3b/68/f96fc79c0b0c6924de4cc737a965de3744ff4ae585a6decad0
Successfully built h2o
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [h2o]1/2 [h2o]
Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.20" 2026-07-21; OpenJDK Runtime Environment Homebrew (build 17.0.20+0); OpenJDK 64-Bit Server 

H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 17 days
H2O_cluster_name:,H2O_from_python_lannaces_ckcfg4
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
21:08:49.149: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

===== LEADERBOARD =====
model

## Autogluon

In [6]:
# =========================
# INSTALAR AUTOGLUON
# =========================
#%pip install autogluon

# =========================
# IMPORTS
# =========================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from autogluon.tabular import TabularDataset, TabularPredictor

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

train_data = TabularDataset(train_df)
test_data = TabularDataset(test_df)

# =========================
# AUTOGLUON
# =========================
predictor = TabularPredictor(
    label='Class',
    eval_metric='f1'
).fit(
    train_data=train_data,
    presets='medium_quality',
    time_limit=300  # Orçamento de tempo em segundos
)

# =========================
# LEADERBOARD
# =========================
print("\n===== LEADERBOARD =====")
leaderboard = predictor.leaderboard(test_data, display=True)

# =========================
# MÉTRICAS DE TODOS OS MODELOS
# =========================
resultados = []

X_test_ag = test_df.drop(columns=['Class'])
y_true = test_df['Class'].values

# Iterar sobre todos os modelos gerados (incluindo Ensembles)
for model_name in predictor.model_names():

    # Previsão específica para cada modelo
    preds = predictor.predict(X_test_ag, model=model_name)
    y_pred = preds.astype(int).values

    # Métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)

    resultados.append({
        "Modelo": model_name,
        "Tipo": model_name.split('_')[0],
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

# =========================
# RESULTADO FINAL
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="F1-score", ascending=False))

No path specified. Models will be saved in: "AutogluonModels/ag-20260929_202029"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.61 GB / 16.00 GB (16.3%)
Disk Space Avail:   235.72 GB / 460.38 GB (51.2%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/TCC/AutogluonModels/ag-20260929_202029"
Train Data Rows:    427652
Train Data Columns: 30
Label Column:       Class
Auto


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch    0.783019   1.000000          f1        0.213071       0.023949  48.013756                 0.213071                0.023949          48.013756            1       True          6
1             CatBoost    0.704918   0.999542          f1        0.015976       0.003096  12.404469                 0.015976                0.003096          12.404469            1       True          3
2              XGBoost    0.661479   0.999771          f1        0.045918       0.005782   4.050798                 0.045918                0.005782           4.050798            1       True          5
3      NeuralNetFastAI    0.617647   1.000000          f1        0.136240       0.008048  62.232897                 0.136240                0.008048          62.23

## PyCaret

In [8]:
# =========================
# INSTALAR PYCARET
# =========================
#%pip install pycaret

# =========================
# IMPORTS
# =========================
import pandas as pd

from sklearn.model_selection import train_test_split
from pycaret.classification import setup, compare_models, predict_model, pull
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# DATAFRAME FINAL
# =========================
train_df = pd.DataFrame(X_train)
train_df['Class'] = y_train.values

test_df = pd.DataFrame(X_test)
test_df['Class'] = y_test.values

# =========================
# PYCARET
# =========================

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

exp = setup(
    data=train_df,
    test_data=test_df,
    target="Class",
    index=False,  # PyCaret assigns a fresh index
    session_id=42,
    verbose=False
)


# Retorna os 10 melhores modelos treinados
modelos_treinados = compare_models(n_select=10)

# =========================
# LEADERBOARD INTERNO DO PYCARET
# =========================
print("\n===== LEADERBOARD (Validação PyCaret) =====")
print(pull())

# =========================
# MÉTRICAS DE TODOS OS MODELOS (No Test Set)
# =========================
resultados = []

for model in modelos_treinados:
    model_name = model.__class__.__name__

    # Previsão no conjunto de teste
    preds_df = predict_model(model, data=test_df, verbose=False)

    y_true = test_df['Class'].values
    # No PyCaret 3.x, a coluna com o rótulo previsto é 'prediction_label'
    y_pred = preds_df['prediction_label'].astype(int).values

    # Métricas
    acc = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
    f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)

    resultados.append({
        "Modelo": model_name,
        "Tipo": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

# =========================
# RESULTADOS
# =========================
df_resultados = pd.DataFrame(resultados)

print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
print(df_resultados.sort_values(by="F1-score", ascending=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9999,1.0000,1.0000,0.9998,0.9999,0.9998,0.9998,38.3710
et,Extra Trees Classifier,0.9999,1.0000,1.0000,0.9999,0.9999,0.9999,0.9999,4.2360
lightgbm,Light Gradient Boosting Machine,0.9996,0.9999,0.9999,0.9993,0.9996,0.9992,0.9992,1.5590
dt,Decision Tree Classifier,0.9987,0.9987,0.9993,0.9982,0.9988,0.9975,0.9975,4.5400
gbc,Gradient Boosting Classifier,0.9900,0.9995,0.9860,0.9944,0.9902,0.9800,0.9800,62.3060
knn,K Neighbors Classifier,0.9887,0.9986,0.9936,0.9843,0.9890,0.9773,0.9774,21.9010
ada,Ada Boost Classifier,0.9842,0.9989,0.9802,0.9887,0.9844,0.9683,0.9683,12.4600
lr,Logistic Regression,0.9807,0.9978,0.9722,0.9899,0.9810,0.9615,0.9616,5.9550
qda,Quadratic Discriminant Analysis,0.9681,0.9921,0.9611,0.9761,0.9685,0.9361,0.9363,0.3980
ridge,Ridge Classifier,0.9389,0.9902,0.8925,0.9866,0.9372,0.8780,0.8822,0.1300



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9999  1.0000  1.0000  0.9998   
et                 Extra Trees Classifier    0.9999  1.0000  1.0000  0.9999   
lightgbm  Light Gradient Boosting Machine    0.9996  0.9999  0.9999  0.9993   
dt               Decision Tree Classifier    0.9987  0.9987  0.9993  0.9982   
gbc          Gradient Boosting Classifier    0.9900  0.9995  0.9860  0.9944   
knn                K Neighbors Classifier    0.9887  0.9986  0.9936  0.9843   
ada                  Ada Boost Classifier    0.9842  0.9989  0.9802  0.9887   
lr                    Logistic Regression    0.9807  0.9978  0.9722  0.9899   
qda       Quadratic Discriminant Analysis    0.9681  0.9921  0.9611  0.9761   
ridge                    Ridge Classifier    0.9389  0.9902  0.8925  0.9866   
lda          Linear Discriminant Analysis    0.9389  0.9902  0.8925  0.9866   
nb     

In [8]:
import sys, pycaret, sklearn
print(sys.executable)
print("PyCaret:", pycaret.__version__)
print("scikit-learn:", sklearn.__version__)

/Users/lannaces/Documents/TCC/.venv311/bin/python
PyCaret: 3.3.2
scikit-learn: 1.9.1
